# MLSys-2026 一键跑分（Google Colab）

本机是 Intel Arc，跑不了 CUDA / Triton，所以用 Colab 的免费 NVIDIA GPU（T4）来产出实测数据。

## 使用步骤
1. 菜单 **代码执行程序 → 更改运行时类型 → 硬件加速器选 T4 GPU**，保存
2. 第 3 个单元格是「环境确认」，先跑一遍，确认 `torch.cuda.is_available()` 是 `True`
3. 第 4 / 5 个单元格是**二选一**的获取代码方式：
   - **路线 A（推荐，最终也要交 GitHub）**：把 `MLSys-2026` 推到 GitHub Public 仓库，在路线 A 里填上仓库地址后运行
   - **路线 B（最快）**：直接把 `MLSys-2026-colab.zip` 上传上来解压
4. 继续往下跑：Task 1 编译 + 跑分 + 画图 → Task 2 全部实验 → 打包下载 `results/`
5. 把下载到的 zip 解压，把 `environment.txt` 和 CSV 贴回 task1 / task2 的 `report.md` 里所有 **[待填]** 位置

> Colab 免费层是 T4（sm_75），脚本里 `-arch=native` 会自动对上；已内置 sm_70 兜底。

In [ ]:
# 1. 环境确认：必须看到 NVIDIA GPU
!nvidia-smi!nvcc --versionimport sys, torchprint('python :', sys.version.split()[0])print('torch  :', torch.__version__, '| cuda:', torch.version.cuda)print('available:', torch.cuda.is_available())print('device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -- 请先切换运行时为 T4 GPU')try:    import triton    print('triton :', triton.__version__)except Exception as exc:    print('triton : not installed ->', exc)

In [ ]:
# 2-A. 路线 A：从 GitHub 拉取（把地址换成你自己的仓库）
REPO_URL = 'https://github.com/USERNAME/MLSys-2026.git'import osif os.path.isdir('MLSys-2026/.git'):    !cd MLSys-2026 && git pull --ff-onlyelse:    !git clone {REPO_URL} MLSys-2026%cd MLSys-2026!ls

In [ ]:
# 2-B. 路线 B：不用 GitHub，直接选中本机的 MLSys-2026-colab.zip 上传（与路线 A 二选一）
from google.colab import files as colab_filesimport zipfile, osuploaded = colab_files.upload()for name in uploaded:    print('uploaded:', name)    if name.lower().endswith('.zip'):        with zipfile.ZipFile(name) as zf:            zf.extractall('.')        print('extracted ->', zf.namelist()[:5], '...')for candidate in ('MLSys-2026', 'MLSys-2026-colab', os.path.splitext(name)[0]):    if os.path.isdir(candidate):        %cd {candidate}        break%cd!ls

In [ ]:
# 3. Task 1：编译全部 .cu，跑完 Vector Add 与三个 GEMM 变体
#    产物：task1/results/*.csv 与 environment.txt（自动记录 nvidia-smi / nvcc / 编译参数）
!bash task1/benchmark/build_and_run.sh

In [ ]:
# 4. Task 1：画图（Shape / Tile Size 对性能的影响）
!pip -q install pandas matplotlib!python task1/benchmark/plot_results.pyimport globfrom IPython.display import Image, displayfor p in sorted(glob.glob('task1/results/figures/*.png')):    print(p)    display(Image(p))

In [ ]:
# 5. Task 2：Triton Vector Add / GEMM / 72 配置 Autotuning
#    Colab 一般自带 triton；若报 ModuleNotFoundError，去掉下面这行的注释再跑
# !pip -q install triton%cd task2!python triton/bench_all.py

In [ ]:
# 6. 打包 results 并下载回本机
%cd ..import shutilshutil.make_archive('mlsys_results_t1', 'zip', root_dir='.', base_dir='task1/results')shutil.make_archive('mlsys_results_t2', 'zip', root_dir='.', base_dir='task2/results')from google.colab import filesfiles.download('mlsys_results_t1.zip')files.download('mlsys_results_t2.zip')

## 拿到数据之后

把 `environment.txt` 和 CSV 发我，我帮你把 `task1/report.md`、`task2/report.md` 里所有
**[待填]** 的表格补齐，并按报告里已写好的框架补上基于实测的解释：

- Block Size 增大后吞吐为何不再上升、1024 是否反而更慢
- TILE 8 / 16 / 32 与方阵 / 小 M Shape 的性能差异
- Register Tiling 的收益与 Register Pressure 的代价
- Autotune 最佳配置在 held-out Shape 上的迁移性